# 04B 记住什么，怎样继续

现在关闭昨天的程序，重新启动一个进程。“记忆”从哪里来？本课把它具体实现为：**明确写入记录 → 按学习者读取 → 检查版本 → 放入下一次请求**。

本 Notebook 使用课程自编学习记录，默认不在录制时重复发送 API 请求。网络实验读取本次课前预跑保存的真实 DeepSeek 回答，同时保留重新调用的完整代码。

**贯穿案例**

沿用同一位学生的中断任务，检查两种实际风险：重启后找不回待解题目，以及摘要把 0.25 错记成 0.9。`MemoryStore` 负责按学习者和资料版本保存、读取与拒绝过期记录；三组已保存的模型请求用于比较输入成本和回答。自编记录是受控样例，不是真实学生数据。


## 学习路线与配套课件

本 Notebook 可以独立从头运行。先读定义和输入，预测结果，再执行并解释差异。小算例帮助理解机制；真实训练、工具执行与保存的 API 记录会分别标明来源。

本节重点：字段完整，却把原题记错了；用服务用量校验字符预算的误差。

课件页：L04.03-S01、L04.03-S03、L04.06-S01、L04.07-S01、L04.07-S03。

沿原有课程单元顺序完成实验；新增对照放在相关代码后。练习先自行作答，展开参考分析后再比较。

In [1]:
from pathlib import Path
import sys, json, copy, hashlib, os, re, tempfile
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
CHAPTER = ROOT / "04 上下文工程与记忆"
sys.path.insert(0, str(CHAPTER / "代码"))
from context_lab import (json_chars, atomic_message_groups, pack_context,
    MEMORY_FIELDS, validate_memory, MemoryStore, TUTOR_SYSTEM, context_variants)
fixture = json.loads((CHAPTER / "数据/learning-session.json").read_text(encoding='utf-8'))
print("使用课程自编学习记录；不读取真实学生资料。")


使用课程自编学习记录；不读取真实学生资料。


## 1. 结构化记录有明确字段

一份摘要可能读起来顺畅，却漏掉“只给提示”这样的条件。先把本任务不能丢的信息写成字段，检查缺失和类型错误，再谈自动总结。

下面的验证只能检查格式与字段，不能证明记录与原始对话一致。程序要做内容核验，仍需要回查来源或额外检查。


In [2]:
print("本课要求的字段:", sorted(MEMORY_FIELDS))
def validate_memory(record):
    if not isinstance(record, dict) or set(record) != MEMORY_FIELDS:
        raise ValueError('学习记录必须完整包含约定的四个字段')
    for field in ['goal', 'pending_question', 'source_version']:
        if not isinstance(record[field], str) or not record[field].strip() or len(record[field]) > 1000:
            raise ValueError('学习记录文本为空、类型错误或过长')
    if not isinstance(record['hints_only'], bool):
        raise ValueError('hints_only 必须为布尔值')
    return copy.deepcopy(record)


本课要求的字段: ['goal', 'hints_only', 'pending_question', 'source_version']


## 2. 保存到磁盘，才可以跨进程读取

下面的记录库以学习者 ID 区分文件。每次先写临时文件，再替换目标文件，避免正常读者看到只写到一半的 JSON。它不是完整数据库，也没有实现多进程并发修改冲突处理。

记录库不是模型的一部分：保存 JSON 不会改变模型参数，只有后来读出并放入请求，模型才可能利用它。我们把产物写到课程忽略目录，不改教学原始资料。


In [3]:
class MemoryStore:
    """课程自编记录库；显式写入/读取/删除，不是自动学习的模型参数。"""
    def __init__(self, root):
        self.root = Path(root).resolve()
        self.root.mkdir(parents=True, exist_ok=True)

    def path(self, learner):
        if not isinstance(learner, str) or not re.fullmatch(r'[a-z0-9_-]{1,40}', learner):
            raise ValueError('学习者 ID 只允许小写字母、数字、下划线和短横线')
        path = self.root / (learner + '.json')
        if path.is_symlink():
            raise ValueError('学习记录不允许使用符号链接')
        return path

    def save(self, learner, record):
        record = validate_memory(record)
        destination = self.path(learner)
        temporary = None
        try:
            with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', dir=self.root,
                                             prefix='.memory-', suffix='.tmp', delete=False) as handle:
                temporary = Path(handle.name)
                json.dump(record, handle, ensure_ascii=False, indent=2)
                handle.flush()
                os.fsync(handle.fileno())
            os.replace(temporary, destination)
        finally:
            if temporary is not None and temporary.exists():
                temporary.unlink()
        return {'learner': learner, 'saved': True}

    def load(self, learner, source_version):
        path = self.path(learner)
        if not path.exists():
            return {'status': 'missing', 'record': None}
        if path.stat().st_size > 16000:
            raise ValueError('学习记录超过教学大小限制')
        record = validate_memory(json.loads(path.read_text(encoding='utf-8')))
        if record['source_version'] != source_version:
            return {'status': 'stale', 'record': None}
        return {'status': 'available', 'record': record}

    def forget(self, learner):
        path = self.path(learner)
        existed = path.exists()
        path.unlink(missing_ok=True)
        return {'learner': learner, 'removed': existed}


In [4]:
RUN_DIR = ROOT / ".local/student-runs/l04/memory"
store = MemoryStore(RUN_DIR)
store.save("student-a", fixture["memory"])
# 新对象重新读取磁盘，不依赖上一个对象的 Python 内存。
reopened = MemoryStore(RUN_DIR)
loaded = reopened.load("student-a", source_version="loss-exercise-v1")
print("读取状态:", loaded["status"])
print(json.dumps(loaded["record"], ensure_ascii=False, indent=2))
assert loaded["record"] == fixture["memory"]
print("另一个学习者:", reopened.load("student-b", "loss-exercise-v1"))


读取状态: available
{
  "goal": "理解自然对数交叉熵",
  "pending_question": "标准答案 Token 的概率为 0.25，计算单个位置的 Loss。",
  "hints_only": true,
  "source_version": "loss-exercise-v1"
}
另一个学习者: {'status': 'missing', 'record': None}


## 3. 过期记录不能静默复用

昨天的练习版本与今天的资料版本可能不同。本课版本不一致时返回 stale，不自动把旧题目塞进新请求；这是一种保守的教学策略，实际产品也可以提供迁移或重新确认过程。

版本相同也不保证记录正确，它只排除一种容易发现的不一致。学习者 ID 的隔离在本课由调用程序负责，不等同于已经实现了真实账户鉴权。


In [5]:
print("版本改变:", reopened.load("student-a", "loss-exercise-v2"))
print("无此学习者:", reopened.load("new-student", "loss-exercise-v1"))
try:
    reopened.load("../outside", "loss-exercise-v1")
except ValueError as error:
    print("路径不合规:", error)
# 这只是目录范围约束，不是完整的用户身份认证系统。


版本改变: {'status': 'stale', 'record': None}
无此学习者: {'status': 'missing', 'record': None}
路径不合规: 学习者 ID 只允许小写字母、数字、下划线和短横线


### 练习：保存另一位学习者，再忘记记录

为 student-b 写入一条新的未完成题目，确认没有改动 student-a。随后删除 student-b，再读取。请解释：删除磁盘记录是否会撤回已经发送给模型的内容？

答案应指出，后续读取和已有请求是不同阶段；删除本地记录不会自动撤回之前的网络请求。


In [6]:
student_record = copy.deepcopy(fixture["memory"])
# TODO：修改未完成题目，保存 student-b，读取并删除它。
print(student_record)


{'goal': '理解自然对数交叉熵', 'pending_question': '标准答案 Token 的概率为 0.25，计算单个位置的 Loss。', 'hints_only': True, 'source_version': 'loss-exercise-v1'}


In [7]:
other = copy.deepcopy(fixture["memory"])
other["pending_question"] = "概率为 0.5，计算单位置 Loss。"
reopened.save("student-b", other)
assert reopened.load("student-a", "loss-exercise-v1")["record"] == fixture["memory"]
assert reopened.load("student-b", "loss-exercise-v1")["record"] == other
print(reopened.forget("student-b"))
print(reopened.load("student-b", "loss-exercise-v1"))


{'learner': 'student-b', 'removed': True}
{'status': 'missing', 'record': None}


## 摘要丢失条件的反例

先检查字段是否齐全，再回到原始对话核对内容。两种检查不能互相替代。


In [8]:
bad_summary = {"goal":"学生在学习 Loss", "source_version":"loss-exercise-v1"}
print("摘要漏了:", sorted(MEMORY_FIELDS - set(bad_summary)))
try:
    validate_memory(bad_summary)
except ValueError as error:
    print("格式检查拒绝:", error)
# 下面这个记录格式完整，但内容错误；字段检查发现不了。
wrong_but_complete = copy.deepcopy(fixture["memory"])
wrong_but_complete["pending_question"] = "标准答案概率为 0.9，计算 Loss。"
validate_memory(wrong_but_complete)
print("完整字段仍可能与原文不符:", wrong_but_complete["pending_question"])


摘要漏了: ['hints_only', 'pending_question']
格式检查拒绝: 学习记录必须完整包含约定的四个字段
完整字段仍可能与原文不符: 标准答案概率为 0.9，计算 Loss。


### 字段完整，却把原题记错了

配套课件：L04.06-S01、L04.07-S01。

上一段已构造字段齐全但把概率 0.25 记成 0.9 的记录。现在计算两种输入各自对应的正确 Loss：即使模型计算没错，上游记忆改变了事实，最后仍会答错原题。

In [9]:
import math
case_original_p, case_wrong_p = 0.25, 0.9
validate_memory(wrong_but_complete)
print("原题:", fixture["memory"]["pending_question"])
print("错误记忆:", wrong_but_complete["pending_question"])
for case_label, case_p in [("原题", case_original_p), ("错误记忆", case_wrong_p)]:
    print(case_label, "p =", case_p, "Loss =", round(-math.log(case_p), 4))
assert not math.isclose(-math.log(case_original_p), -math.log(case_wrong_p))


原题: 标准答案 Token 的概率为 0.25，计算单个位置的 Loss。
错误记忆: 标准答案概率为 0.9，计算 Loss。
原题 p = 0.25 Loss = 1.3863
错误记忆 p = 0.9 Loss = 0.1054


**结果解读**

原题 Loss 约 1.3863，错误记忆对应 0.1054。字段校验通过不能证明内容忠实；遇到错误答案，应先核对发给模型的题目和条件，再判断计算或模型解释是否出错。

**小练习**

若只保存“学生在学 Loss”，缺少哪些继续解题的信息？用原题和当前困难各举一项。

<details><summary>完成后展开参考分析</summary>

至少要保留待解题目中的概率与目标，以及最近尝试/困难；版本用于确认这条状态仍属于当前资料。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

## 4. 先核对对照实验实际改了什么

我们比较完整历史、机械保留最近两条历史、结构化学习记录。系统指令与当前问题相同，改变的是中间提供的信息。三个方案只是同一小场景的输入组织比较，**不是长上下文基准，也没有复现 Lost in the Middle**。

完整历史和结构化记录都含题目；最近两条不含题目，所以合理结果可能是请求澄清。要求三个方案都直接完成原题，反而可能奖励猜测。

本实验的成功条件：有题目时给对应提示并保留“不给数值答案”；信息缺失时说明需要补充。接下来检查实际回答，不预先写死结果。


In [10]:
variants = context_variants(fixture)
for name, messages in variants.items():
    text = json.dumps(messages, ensure_ascii=False)
    print(name, "JSON 字符:", json_chars(messages), "含原题概率:", "0.25" in text)
assert len({m[0]["content"] for m in variants.values()}) == 1
assert len({m[-1]["content"] for m in variants.values()}) == 1


full_history JSON 字符: 564 含原题概率: True
recent_only JSON 字符: 283 含原题概率: False
curated_memory JSON 字符: 381 含原题概率: True


## 5. 阅读真实回答，或者显式重跑

默认从本次课前预跑的真实 API 记录读取，先核对资料哈希和请求消息。记录里的模型名只说明本次运行条件，不代表以后仍可用；保存结果也不意味着每次调用都会相同。

若要重跑，在本地配置 `DEEPSEEK_API_KEY` 后把开关改为 True。程序会先打印本次选择和当前账号可用的模型，并拒绝使用已下线的名称。下面会发出三次真实请求，记录服务返回的 prompt_tokens；不要把上一份 Notebook 的字符计数当成 Token。


In [11]:
sys.path.insert(0, str(ROOT / "02 从大模型到 Agent 组成与最小循环/代码"))
from agent_lab import DeepSeekModel, ModelError, list_deepseek_models
RUN_LIVE = False
selected_model = os.environ.get("DEEPSEEK_MODEL", "deepseek-flash")
print("本次选择的模型:", selected_model)
if RUN_LIVE:
    available_models = list_deepseek_models()
    print("当前账号可用模型:", available_models)
    if selected_model not in available_models:
        raise ModelError("所选模型不在当前账号的可用列表中；请先更新 DEEPSEEK_MODEL")
    records = []
    for name, messages in variants.items():
        model = DeepSeekModel(selected_model)
        response = model(messages, [])
        records.append({"name":name, "messages":messages, "response":response, "usage":model.usage})
else:
    saved = json.loads((CHAPTER / "数据/实验记录/deepseek-context.json").read_text(encoding='utf-8'))
    assert saved["fixture_sha256"] == hashlib.sha256((CHAPTER / "数据/learning-session.json").read_text(encoding="utf-8").replace("\r\n", "\n").encode("utf-8")).hexdigest()
    records = saved["variants"]
    for record in records:
        assert record["messages"] == variants[record["name"]]
print("结果来源:", "本次在线请求" if RUN_LIVE else "已保存真实 API 记录")
if not RUN_LIVE:
    print("保存记录:", saved["model"], saved["recorded_at"])
for record in records:
    print("\n方案:", record["name"])
    print(record["response"]["content"])
    print("服务返回的输入 Token:", record["usage"][0]["prompt_tokens"])


本次选择的模型: deepseek-flash
结果来源: 已保存真实 API 记录
保存记录: deepseek-flash 2026-09-24T05:45:09.439166+00:00

方案: full_history
下一步：写出单位置交叉熵的公式，并把“标准答案 Token 的概率”代入对应的概率项。
服务返回的输入 Token: 200

方案: recent_only
请先提供昨天练习的题目或你做到哪一步，我才能给出下一步。
服务返回的输入 Token: 102

方案: curated_memory
下一步：写出交叉熵公式 Loss = -ln(p)，把 p=0.25 代入后先判断结果正负，再继续化简；先不要算最终数值。
服务返回的输入 Token: 139


### 用服务用量校验字符预算的误差

配套课件：L04.03-S01、L04.03-S03、L04.07-S03。

这三条回答来自同一次真实 DeepSeek 对照。用 pandas 把本地 JSON 字符数、服务返回的输入 Token、缓存命中与完整回答放到一张表；字符数只能帮助本地装箱，不能换算成可靠的计费 Token。

In [12]:
import pandas as pd
case_rows = []
for item in records:
    usage_row = item["usage"][0]
    case_rows.append({"方案": item["name"], "JSON字符": json_chars(item["messages"]),
        "输入Token": usage_row["prompt_tokens"],
        "缓存命中Token": usage_row.get("prompt_cache_hit_tokens", 0),
        "回答": item["response"]["content"]})
case_frame = pd.DataFrame(case_rows).set_index("方案")
print(case_frame[["JSON字符", "输入Token", "缓存命中Token"]].to_string())
assert len(case_frame) == 3 and case_frame["输入Token"].gt(0).all()
assert case_frame["JSON字符"].nunique() > 1
print("请逐条读回答：信息缺失时是否澄清，信息完整时是否遵守只给提示？")


                JSON字符  输入Token  缓存命中Token
方案                                        
full_history       564      200          0
recent_only        283      102          0
curated_memory     381      139          0
请逐条读回答：信息缺失时是否澄清，信息完整时是否遵守只给提示？


**结果解读**

表中的两个单位不能相互代换。缓存命中是服务计量，不证明回答正确；输入 Token 较少也可能因为删掉了原题。三个回答需要按任务契约逐条人工复核。

**小练习**

把最短输入直接判为最佳方案会犯什么错误？指出它少了哪个事实，并检查模型是否请求澄清。

<details><summary>完成后展开参考分析</summary>

最近消息方案可能少了概率题与辅导偏好。应同时看实际消息、回答和服务用量；缺题时主动澄清可以比猜出数值更符合契约。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

## 6. 少发 Token 与答得正确分开评

先逐项核对：题目是否对上、有没有直接给出最终数值、信息缺失时有没有澄清。再比较实际输入 Token。不能把“输入更少”直接当成“教学效果更好”。

下面的数字匹配只是帮助定位，不是完整评分器。模型可能用另一种写法泄露答案，也可能在引用错误例子时提到某数字，仍需读完整回答。

Lost in the Middle 研究了当时模型在特定检索任务中的位置效应；不能把它的结果当作所有 2026 年模型、所有任务的定律。这里的例子短得多，主要验证信息有没有被我们自己丢掉。


In [13]:
for record in records:
    answer = record["response"]["content"]
    print(record["name"], {
        "提到原题概率": "0.25" in answer,
        "出现数值答案片段_需人工核对": "1.386" in answer,
        "输入Token": record["usage"][0]["prompt_tokens"],
    })
# 不根据这几个字符串直接宣布任务成功；第七节会正式设计评分器。


full_history {'提到原题概率': False, '出现数值答案片段_需人工核对': False, '输入Token': 200}
recent_only {'提到原题概率': False, '出现数值答案片段_需人工核对': False, '输入Token': 102}
curated_memory {'提到原题概率': True, '出现数值答案片段_需人工核对': False, '输入Token': 139}


## 7. 交付：记忆与回答审计报告

最后把记忆字段、保存位置、模型运行条件、三种输入和人工核验提示汇总成一个审计包。报告保留原始回答和服务返回的用量，不把字符串匹配直接写成“通过”。

这样下一位开发者可以看到：哪条记录被读取、哪种输入包含原题、模型当时叫什么、回答还需要人工核对什么。


In [14]:
RUN_DIR = ROOT / ".local/student-runs/l04"
RUN_DIR.mkdir(parents=True, exist_ok=True)
recorded_model = selected_model if RUN_LIVE else saved["model"]
comparison_audit = []
for record in records:
    answer = record["response"]["content"]
    comparison_audit.append({
        "variant": record["name"],
        "contains_original_question": "0.25" in json.dumps(record["messages"], ensure_ascii=False),
        "prompt_tokens": record["usage"][0]["prompt_tokens"],
        "response": answer,
        "review_hints_not_final_grades": {
            "mentions_original_probability": "0.25" in answer,
            "contains_numeric_answer_fragment": "1.386" in answer,
            "asks_for_more_information": any(word in answer for word in ["补充", "提供", "题目"]),
        },
    })

memory_audit = {
    "artifact": "L04-memory-and-response-audit",
    "source_record": "04 上下文工程与记忆/数据/实验记录/deepseek-context.json" if not RUN_LIVE else "new_live_run",
    "model_name_at_run": recorded_model,
    "memory_fields": sorted(MEMORY_FIELDS),
    "memory_store": str(RUN_DIR / "memory"),
    "comparisons": comparison_audit,
    "human_review_required": [
        "answer_matches_the_actual_question",
        "answer_respects_hint_only_preference",
        "missing_information_is_not_guessed",
        "memory_facts_match_the_original_source",
    ],
}
audit_json = RUN_DIR / "memory-and-response-audit.json"
audit_json.write_text(json.dumps(memory_audit, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
audit_md = RUN_DIR / "memory-and-response-audit.md"
lines = [
    "# L04 记忆与回答审计",
    "",
    f"- 运行模型：`{recorded_model}`",
    f"- 记忆字段：{', '.join(sorted(MEMORY_FIELDS))}",
    "- 结论边界：下列自动提示用于定位，最终判断仍需阅读完整回答并回查原始记录。",
    "",
]
for item in comparison_audit:
    lines.extend([
        f"## {item['variant']}",
        "",
        f"- 输入包含原题：{item['contains_original_question']}",
        f"- 输入 Token：{item['prompt_tokens']}",
        f"- 自动提示：{json.dumps(item['review_hints_not_final_grades'], ensure_ascii=False)}",
        "",
    ])
audit_md.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("已生成审计报告:", audit_json.relative_to(ROOT), "和", audit_md.relative_to(ROOT))
print("模型名用于复现当时条件；未来重跑请先列出当前账号可用模型。")


已生成审计报告: .local\student-runs\l04\memory-and-response-audit.json 和 .local\student-runs\l04\memory-and-response-audit.md
模型名用于复现当时条件；未来重跑请先列出当前账号可用模型。


## 本节回顾

你已实际做过：查看请求、按预算选择完整消息组、保存并重读学习记录、拒绝过期记录、比较三种输入下的真实回答。

上下文工程关注“这一步看见什么”；记忆关注“什么信息以后还能取回”。下一节引入 MCP 和 Skills，分别扩展可调用能力与可复用方法；第六节再把这些组件放进完整运行系统。

课后提交：04A 的上下文选择清单，以及 04B 的记忆与回答审计 JSON/Markdown。两份产物都来自课程自编资料和已保存结果，默认网络关闭。

参考：[Anthropic 上下文工程](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)、[OpenAI Agents SDK 上下文管理](https://openai.github.io/openai-agents-python/context/)、[Lost in the Middle，arXiv v3](https://arxiv.org/abs/2307.03172v3)。核验日期：2026-09-22。
